# Intro to virtual brain model testing

This is an introductory chapter to testing and comparing the predictive equivalence of two models using different simulators. It aims to streamline this process and remove the need for a deep understanding of how the models are designed.

Why is this useful? There are many models and simulators already out there, but they may have some limitations. For example, the concept of multiple connectivities is completely foreign to the TVB library. So, developers often write new implementations of existing models when they encounter these limitations. However, software engineers know that untested software is not correctly written software, and as such arises the need to test them.

In this notebook we'll introduce key concepts that are necessary to build a correct test suite. There are relatively few steps required, and we've built up a solid supporting foundation for creating such a test suite. Should you follow it and implement the tests, your models can be considered equivalent.

## The requirements

Firstly, you need a ground truth model. For these tutorials we've chosen TVB as it is the both mature and flexible. However, we're also showing examples using both VBJax and Neurolib, so if you wish or it suits you better, you may use those. Even if you're comparing two simulators not mentioned in these notebooks, this guide should be sufficient to write the necesarry wrappers.

The second requirement is for your tested simulator to be able to do exactly one step of Euler integration. The main drawback of Euler is that it accumulates errors quickly. That drawback is minimized by doing only one step. More complex integrators, like Heun, utilize a "correction step" which gives a more mathematically sound approximation. However, that is the exact reason why they are not suitable for use in this test suite, as it is another calculation which can be inconsistent between the models. The inconsistencies then complicate determining which discrapancies are caused by the integrator and which are caused by model differences. Euler provides a clean deterministic calculation.

The last necessary requirement is the setting of initial conditions. This means values of state variables, and in case you want to test delays in your model, also some sort of history.

The setting of other values, such as dfun parameters, the time step dt, conduction speed, the connectivity matrix, and the tract‑length matrix is optional, but when used they must be configured identically for both models. So at least one of the models is fully configurable, but ideally both.

## The limitations

As with any testing, there is only so much we can test, and for the rest we'll assume that it's working correctly. Some simulators provide a unit test suite for their own code, and we rely on those unit tests to handle parts of the simulators we're not examining.

In the requirements section, for example, we mentioned Heun integration and why we won't be testing it. However, should you wish to test that Heun, or any other integrator, is correct, you may do so by comparing sufficiently precise Euler integration against Heun results. As the main benefit of Heun is calculation speed, a one time comparison that is resource-intensive seems like a worthwhile investment. This is one of the unit tests mentioned previously that we expect to be handled within the test suite of a given simulator itself.

The brain network models have numerous parameter and of course it is not feasible to test all possible combinations of their values. Here we take a sampling approach informed by apriory knowledge about the dynamical regimes of the model, and describe it in detail in the next section.

## The key concepts

The following four concepts are the core of our testing approach. By covering each one we can declare two models effectively equivalent:
- implementation of mathematical formulas describing internal behavior of an isolated node (deterministic part)
- network interaction (connectivity and coupling)
- time delays (if present)
- noise handling (stochastic representation of the system)

These ideas will be elaborated on in their individual tutorial notebooks.

## Implementation guidelines

This chapter will show you examples of necessary steps prior to the first test, and explain why they are necessary.

### The configuration

If we make the assumption you have a working knowledge of how virtual brain simulators work, and of the model you're testing, then this is the next step you should make. We're going to show here a mostly complete configuration class, but this guide and adjacent software is designed in such a way that you only need to replace the differential function parameters.

In [ ]:
import numpy as np
import tvb.simulator.lab as tvbl

class Config:
    def __init__(self, initial_conditions_seed, noise_seed=42):
        # dfun parameters
        self.a = 0.35
        self.w = 0.2

        # settings relevant for connectivity testing
        self.coupling_strength = 1.0
        self.speed = 2.0
        self.history_length = 10

        # settings relevant for noise testing
        self.dt = 0.1
        self.noise = 0.0
        self.noise_seed = noise_seed

        # generic conditions required for model initialization
        self.conn = None
        self.init_cond_rng = np.random.default_rng(seed=initial_conditions_seed)
        self.init_cond = None

    def __config_connectivity(self):
        # Import connectivity and tract length matrices
        # You're encouraged to use real world connectivities instead of this one
        if self.conn is None:
            self.conn = tvbl.connectivity.Connectivity().from_file()
        np.fill_diagonal(self.conn.weights, 0)  # remove self-connections
        self.conn.speed = np.r_[self.speed]
        self.size = self.conn.weights.shape[0]

    def init_config_for_connectivity(self):
        # Will be shown in connectivity testing notebook
        pass

    def init_config_for_delays(self):
        # Will be shown in connectivity testing notebook
        pass

    def init_cond_for_noise(self):
        # Will be shown in noise testing notebook
        pass

    def get_good_history_shape(self):
        # Will be shown in noise testing notebook
        pass


/home/dj/diplomka/model_testing/.venv/lib/python3.12/site-packages/tvb/datatypes/surfaces.py:60: UserWarning: Geodesic distance module is unavailable; some functionality for surfaces will be unavailable.
  warnings.warn(msg)


### The wrappers

With an object that holds configuration for both models, we need to write a way for the simulator to accept this configuration. Thus, a wrapper. The second function of the wrapper is to return results in a comparable across simulators.

Notably, different tests make use of different initial conditions.

Below is the wrapper for a TVB simulator instance of a SupHopf model with linear coupling and additive noise. The return value of `sim.run` is trimmed, as TVB returns run metadata alongside model results.

In [ ]:
import numpy as np
from tvb.simulator import simulator, coupling
from tvb.simulator.integrators import EulerStochastic
from tvb.simulator.monitors import Raw
from tvb.simulator.models.oscillator import SupHopf
import tvb.simulator.lab as tvbl


class TvbModel:
    def __init__(self, config: Config):
        self.config = config
        self._configure_sim()

    def _configure_sim(self):
        self.sim = simulator.Simulator(
            connectivity=self.config.conn,
            model=SupHopf(a=np.r_[self.config.a], omega=np.r_[self.config.w]),
            integrator=EulerStochastic(
                dt=self.config.dt,
                noise=tvbl.noise.Additive(
                    nsig=np.r_[self.config.noise],
                    noise_seed=self.config.noise_seed,
                ),
            ),
            initial_conditions=self.config.init_cond,
            conduction_speed=self.config.speed,
            monitors=[Raw()],
            simulation_length=self.config.dt, # run a single Euler step
            coupling=coupling.Scaling(a=np.r_[self.config.coupling_strength]),
        )
        self.sim.configure()

    def run(self):
        # Return only relevant data from the simulation
        return self.sim.run()[0][1]
